In [1]:
pip install streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 796.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.4 MB/s eta 0:00:00


In [7]:
import streamlit as st
import numpy as np
import pandas as pd
import unicodedata
import joblib
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ──────────────────────────────────────────────────────────────────────────────
# 1) Load artifacts (cached so it only runs once)
@st.cache(allow_output_mutation=True)
def load_artifacts():
    model    = load_model("model.keras")      # your trained Keras model
    scaler   = joblib.load("scaler.pkl")      # fitted StandardScaler
    tab_cols = joblib.load("tab_cols.pkl")    # list of tab_df.columns
    return model, scaler, tab_cols

model, scaler, tab_cols = load_artifacts()


2025-04-30 16:35:54.038 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:35:54.296 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-04-30 16:35:54.297 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:35:54.299 
`st.cache` is deprecated and will be removed soon. Please use one of Streamlit's new
caching commands, `st.cache_data` or `st.cache_resource`. More information
[in our docs](https://docs.streamlit.io/develop/concepts/architecture/caching).

**Note**: The behavior of `st.cache` was updated in Streamlit 1.36 to the new caching
logic used by `st.cache_data` and `st.cache_resource`. This might lead to some problems
or unexpected behavior in certain edge cases.

2025-04-30 16:35:54.302 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
202

TypeError: path should be path-like or io.BytesIO, not <class 'NoneType'>

In [8]:
# ──────────────────────────────────────────────────────────────────────────────
# 2) Build UI
st.title("YouTube Views Predictor")

st.sidebar.header("Tabular Inputs")
likes        = st.sidebar.number_input("Likes", min_value=0, value=1000)
subscribers  = st.sidebar.number_input("Subscribers", min_value=0, value=10000)
publish_hour = st.sidebar.slider("Publish Hour (0–23)", 0, 23, 12)

# cyclic-encode hour
r = 2 * np.pi * publish_hour / 24
hour_sin = np.sin(r)
hour_cos = np.cos(r)

# derive categories from tab_cols
regions    = sorted(c.split("_",1)[1] for c in tab_cols if c.startswith("region_"))
days       = sorted(c.split("_",2)[2] for c in tab_cols if c.startswith("day_of_week_"))
categories = sorted(c.split("_",2)[2] for c in tab_cols if c.startswith("category_id_"))

region      = st.sidebar.selectbox("Region", regions)
day_of_week = st.sidebar.selectbox("Day of Week", days)
category_id = st.sidebar.selectbox("Category ID", categories)

title       = st.sidebar.text_input("Video Title (optional)")
symbol_count = sum(unicodedata.category(c).startswith("So") for c in title)

2025-04-30 16:36:37.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:37.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:38.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:38.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:38.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:38.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:38.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:36:38.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [12]:
# 3) Image uploader
st.header("Upload Thumbnail")
uploaded_file = st.file_uploader("Choose a JPG/PNG", type=["jpg","jpeg","png"])
if not uploaded_file:
    st.warning("Please upload a thumbnail image.")
    st.stop()

img = load_img(uploaded_file, target_size=(128,128))

img_arr = img_to_array(img) / 255.0
st.image(img, use_column_width=True)

2025-04-30 16:40:02.120 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.122 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:02.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [13]:
# 4) Assemble tabular feature vector
row = dict.fromkeys(tab_cols, 0.0)
row.update({
    "likes":         likes,
    "subscribers":   subscribers,
    "hour_sin":      hour_sin,
    "hour_cos":      hour_cos,
    "symbol_count":  symbol_count,
    f"region_{region}":           1.0,
    f"day_of_week_{day_of_week}": 1.0,
    f"category_id_{category_id}": 1.0,
})
X_tab = pd.DataFrame([row], columns=tab_cols)
X_tab_scaled = scaler.transform(X_tab)

# ──────────────────────────────────────────────────────────────────────────────
# 5) Predict
X_img_input = np.expand_dims(img_arr, 0)   # shape (1,128,128,3)
y_log_pred   = model.predict([X_img_input, X_tab_scaled])[0,0]
y_pred       = np.expm1(y_log_pred)        # back to raw views

# ──────────────────────────────────────────────────────────────────────────────
# 6) Display result
st.subheader("Estimated View Count")
st.markdown(f"### **{y_pred:,.0f}** views")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step


<ipython-input-13-e2c871aa75ae>:20: RuntimeWarning: overflow encountered in expm1
  y_pred       = np.expm1(y_log_pred)        # back to raw views
2025-04-30 16:40:06.436 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:06.436 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:06.439 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-04-30 16:40:06.440 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()